In [ ]:
import re
import sys
import asyncio
import concurrent.futures
import pandas as pd
from datetime import datetime
from playwright.sync_api import sync_playwright

# Windows-specific fix for Jupyter
if sys.platform.startswith("win"):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# Configuration
PLACE_URL = "https://www.google.com/maps/place/Me+Gacoan/@-6.7368909,108.5397148,17z/data=!4m16!1m7!3m6!1s0x2e6ee30053665411:0x596def9932dfd843!2sMie+Gacoan+Tuparev!8m2!3d-6.7113559!4d108.5408884!16s%2Fg%2F11w9lfgy67!3m7!1s0x2e6f1dab5e829e43:0x9e14646845070d35!8m2!3d-6.7368909!4d108.5422897!9m1!1b1!16s%2Fg%2F11kpznf2zt?entry=ttu&g_ep=EgoyMDI2MDgyNS4wIKXMDSoASAFQAw%3D%3D"
MAX_REVIEWS = 1000      # Target number of reviews
HEADLESS = False        # Keep False so you can watch the scrolling process
OUTPUT_CSV = "reviews.csv"

def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip() if text else ""

def scrape_ui_robust():
    reviews = {}
    
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=HEADLESS, args=["--disable-blink-features=AutomationControlled"])
        context = browser.new_context(viewport={"width": 1366, "height": 900}, locale="id-ID")
        page = context.new_page()
        
        # Open Google Maps
        print("[1/4] Opening Google Maps...")
        page.goto(PLACE_URL, wait_until="domcontentloaded")
        page.wait_for_timeout(3000)
        
        # Open the Reviews tab
        print("[2/4] Finding and opening the Reviews tab...")
        try:
            page.locator("button[aria-label*='Ulasan'], button[aria-label*='ulasan'], button[aria-label*='Reviews']").first.click(timeout=5000)
        except Exception:
            try:
                page.get_by_role("tab", name=re.compile("Ulasan|Reviews", re.I)).click(timeout=5000)
            except Exception:
                print("Failed to open the Reviews tab! The page structure may not have loaded fully yet.")
                return []
                
        page.wait_for_timeout(3000)
        
        # Extraction and auto-scroll loop
        print("[3/4] Extracting and auto-scrolling...")
        stagnant_rounds = 0
        prev_count = 0
        
        while len(reviews) < MAX_REVIEWS and stagnant_rounds < 12:
            # A. Click "More" buttons so text isn't truncated (optional)
            for btn in page.locator("button[aria-expanded='false']:has-text('Lainnya'), button[aria-expanded='false']:has-text('More')").all():
                try:
                    btn.click(timeout=500)
                except Exception:
                    pass
            
            # B. Capture all review cards currently on screen
            cards = page.locator("div[data-review-id]").all()
            for card in cards:
                try:
                    rev_id = card.get_attribute("data-review-id")
                    if not rev_id or rev_id in reviews:
                        continue
                        
                    # Extract name
                    name = ""
                    try:
                        # Find the first element with text inside the card
                        name = card.locator("button[aria-label]").first.inner_text().split("\n")[0]
                    except Exception:
                        pass
                        
                    # Extract rating
                    rating = None
                    try:
                        aria = card.locator("span[role='img']").first.get_attribute("aria-label")
                        match = re.search(r"(\d+(?:\.\d+)?)", aria.replace(",", "."))
                        if match:
                            rating = float(match.group(1))
                    except Exception:
                        pass
                        
                    # Extract review text (from the element with class containing 'wiI7pd')
                    body = ""
                    try:
                        body = card.locator(".wiI7pd").first.inner_text()
                    except Exception:
                        pass
                        
                    reviews[rev_id] = {
                        "review_id": rev_id,
                        "reviewer_name": clean_text(name),
                        "rating": rating,
                        "review_text": clean_text(body),
                        "scraped_at": datetime.now().isoformat(timespec="seconds")
                    }
                except Exception:
                    continue
            
            # C. Progress check
            current_count = len(reviews)
            print(f"   -> Collected: {current_count} reviews")
            
            if current_count >= MAX_REVIEWS:
                break
                
            if current_count == prev_count:
                stagnant_rounds += 1
            else:
                stagnant_rounds = 0
                
            prev_count = current_count
            
            # D. JAVASCRIPT INJECTION (ABSOLUTE SCROLL)
            # This forcibly finds any element wrapping the reviews and scrolls its 'scrollTop' value to the maximum bottom.
            page.evaluate("""
                () => {
                    const reviewElements = document.querySelectorAll('div[data-review-id]');
                    if (reviewElements.length > 0) {
                        const lastElement = reviewElements[reviewElements.length - 1];
                        lastElement.scrollIntoView(true);
                        
                        let currentElement = lastElement.parentElement;
                        while (currentElement) {
                            if (currentElement.scrollHeight > currentElement.clientHeight) {
                                currentElement.scrollTop = currentElement.scrollHeight;
                            }
                            currentElement = currentElement.parentElement;
                        }
                    }
                }
            """)
            
            page.wait_for_timeout(2500)
            
        browser.close()
        
    print(f"\n[4/4] DONE! Successfully extracted {len(reviews)} reviews.")
    return list(reviews.values())[:MAX_REVIEWS]

# Threaded execution (prevents errors in Jupyter)
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
    results = executor.submit(scrape_ui_robust).result()

# Save results to CSV
df = pd.DataFrame(results)
if not df.empty:
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Data saved to: {OUTPUT_CSV}")
    display(df.head())
else:
    print("Failed to extract reviews. Check whether the Reviews tab opened successfully in the browser.")

[1/4] Opening Google Maps...
[2/4] Finding and opening the Reviews tab...
[3/4] Extracting and auto-scrolling...
   -> Collected: 0 reviews
   -> Collected: 0 reviews
   -> Collected: 3 reviews
   -> Collected: 3 reviews
   -> Collected: 10 reviews
   -> Collected: 20 reviews
   -> Collected: 30 reviews
   -> Collected: 40 reviews
   -> Collected: 50 reviews
   -> Collected: 60 reviews
   -> Collected: 80 reviews
   -> Collected: 90 reviews
   -> Collected: 120 reviews
   -> Collected: 150 reviews
   -> Collected: 160 reviews
   -> Collected: 170 reviews
   -> Collected: 180 reviews
   -> Collected: 190 reviews
   -> Collected: 200 reviews
   -> Collected: 210 reviews
   -> Collected: 220 reviews
   -> Collected: 230 reviews
   -> Collected: 250 reviews
   -> Collected: 260 reviews
   -> Collected: 280 reviews
   -> Collected: 290 reviews
   -> Collected: 300 reviews
   -> Collected: 310 reviews
   -> Collected: 320 reviews
   -> Collected: 330 reviews
   -> Collected: 340 reviews
   -

,review_id,reviewer_name,rating,review_text,scraped_at
0,Ci9DQUlRQUNvZENodHljRjlvT25NMVYxOUNjMmxRVlhsS0...,,2.0,"Rasa mie nya beda, pucettttt ga seperti biasan...",2026-08-31T16:52:13
1,Ci9DQUlRQUNvZENodHljRjlvT213elJteGZOM2RsUkhKQl...,,4.0,"Seperti biasa, mie gacoan selalu ramai pengunj...",2026-08-31T16:52:13
2,Ci9DQUlRQUNvZENodHljRjlvT21KWGJGOXZRbmt5ZFhObU...,,1.0,Dari awal kesini jujur sistem pemesanan nya ri...,2026-08-31T16:52:13
3,Ci9DQUlRQUNvZENodHljRjlvT21OUk4zaEhRMWhWUnpneV...,,1.0,"Mungkin hari sialnya saya, udah sering dateng ...",2026-08-31T16:52:20
4,Ci9DQUlRQUNvZENodHljRjlvT2tsc1Ixa3hWMWhJV1drMl...,,1.0,"padahal antrian lagi rame bgt, waiting list ju...",2026-08-31T16:52:20
